# Matched-training robustness study at 130M parameters

Runs `scripts/scale_study.py` on one GPU (A100 recommended, L4 works).

Every arm starts from the same public checkpoint (`state-spaces/mamba-130m-hf`) and gets the same data, optimiser, steps and seeds:
unconstrained vs. constrained, clean vs. adversarial fine-tuning, plus single-constraint ablations.
Every attack is re-optimised against each fine-tuned checkpoint.

**Runtime → Change runtime type → A100 GPU (or L4).** Run the cells in order. Results are written to Google Drive after every stage, so a disconnected session resumes where it stopped: just run the same cells again.

No Kaggle or Hugging Face token is needed: the checkpoint, tokenizer and WikiText-2 are public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type"
gpu = torch.cuda.get_device_name(0); PRESET = "a100" if "A100" in gpu else "l4"
print("GPU:", gpu, "| preset:", PRESET)

## 1. Output folder on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/lipmamba_scale_study"
import os; os.makedirs(OUT, exist_ok=True); os.environ["OUT"] = OUT; os.environ["PRESET"] = PRESET

## 2. Code

In [ ]:
%cd /content
!rm -rf CV && git clone -q --depth 1 -b claude/implement-lipmamba-model-SHbBY https://github.com/rogerpanel/CV.git
%cd /content/CV/lipmamba
!pip -q install transformers datasets pytest
os.environ["PYTHONPATH"] = "/content/CV/lipmamba/src"
!git log --oneline -1

## 3. Checks (about 2 minutes)
The first command confirms that the converted model computes the same function as the `transformers` Mamba implementation. The second runs the whole pipeline on a toy model.

In [ ]:
!python -m pytest -q tests/test_pretrained.py
!python scripts/scale_study.py run --model lip_at --tiny --out /content/smoke/lip_at_s0.json

## 4. Pilot (about 15 minutes): does fine-tuning work for both architectures?

300 steps, no attacks. Look at the last `step` lines and the `final` line of each run:

* `acc` should approach 1.0 and `final` clean/update accuracy should be high for **both** models;
* `s/step` tells you how long the full run takes (section 5 uses 1500 steps on A100, 1000 on L4).

If the constrained model (`lip`) has not learned the task while `mamba` has, do **not** start section 5. Send me `pilot/*.json`; the comparison would not be meaningful and the settings need adjusting first.

In [ ]:
!python scripts/scale_study.py run --model mamba --seed 0 --preset $PRESET --steps 300 --skip-attacks --out $OUT/pilot/mamba_s0.json
!python scripts/scale_study.py run --model lip   --seed 0 --preset $PRESET --steps 300 --skip-attacks --out $OUT/pilot/lip_s0.json

## 5. Main study, seed 0 (five arms)

`mamba_released`, `mamba`, `lip`, `mamba_at`, `lip_at`. Finished arms are skipped when you re-run the cell.
Expect several hours on an A100; each arm prints its own timing.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 -- --preset $PRESET

## 6. More seeds and the ablations
Run this when section 5 has finished and you still have compute. Three seeds give the error bars reviewers expect; the ablations (seed 0 only) answer the request for an ablation with attack success rate.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 1 2 --ablations -- --preset $PRESET

## 7. Summary and download
Upload `lipmamba_scale_results.zip` (a few hundred kilobytes: JSON files only) to the chat.

In [ ]:
!python scripts/scale_study.py aggregate --dir $OUT/main
!cd $OUT && zip -qr /content/lipmamba_scale_results.zip main pilot -x '*.pt'
from google.colab import files; files.download('/content/lipmamba_scale_results.zip')